In [ ]:
import ast
import pandas as pd
import numpy as np
from pathlib import Path
from collections import defaultdict

import matplotlib.pyplot as plt
plt.style.use("seaborn-v0_8-whitegrid")

from dataset_evaluation.evaluation_framework import EvaluationFramework

## Load datasets

### Baseline

In [ ]:
llama = pd.read_csv('model_test_llama.csv')
llama.rename(columns={"stories": "story"}, inplace=True)

gemma = pd.read_csv('model_test_gemma.csv')
gemma.rename(columns={"stories": "story"}, inplace=True)

phi = pd.read_csv('model_test_phi.csv')
phi.rename(columns={"stories": "story"}, inplace=True)

### ChiSCor

In [ ]:
path_name = 'ChiSCor_master_df.csv'
df_chiscor = pd.read_csv(path_name, index_col=0)
df_chiscor = df_chiscor.rename(columns={'story_raw': 'story'})

### Reference corpora

In [ ]:
ref_standard_dep = Path('datasets/BasiScript/BS_dep_lexicon.csv')
ref_standard_uni = Path('datasets/BasiScript/BS_unigram_lexicon.csv')
ref_standard_bi = Path('datasets/BasiScript/BS_bigram_lexicon.csv')

In [ ]:
ref_spoken_b_csv = Path('datasets/CGN/CGN_pos_bigram.csv')
ref_spoken_u_csv  = Path('datasets/CGN/CGN_pos_unigram.csv')
ref_spoken_t_csv = Path('datasets/CGN/CGN_pos_trigram.csv')

In [ ]:
all_datasets = {'llama': llama, 'gemma': gemma, 'phi': phi, 'chiscor': df_chiscor}

## Story quality

Overview used metrics:
- Coherence
	- Local contextuality
- Grammaticality
	- Grammaticality
- Surprise
	- Creative perplexity
- Diversity
	- Lexical diversity: self-bleu
	- lexical diversity: moving mtld
- Complexity
	- Lexical complexity: unique words
	- Lexical complexity: Average word length
	- Syntactic complexity: avg. components
	- Syntactic complexity: dependency distance
	- Syntactic complexity: syntactic tree depth

In [ ]:
eval_f = EvaluationFramework(language='nl',
							  pos_unigram=ref_spoken_u_csv, 
							  pos_bigram=ref_spoken_b_csv,
							  pos_trigram=ref_spoken_t_csv,
							  ref_unigram=ref_standard_uni,
							  ref_bigram=ref_standard_bi,
							  ref_ling_constrained=ref_standard_dep,
							  embedding_model='jegormeister/bert-base-dutch-cased')

In [ ]:
# Surprise: creative perplexity
eval_f.add_pipe('creative_perplexity_dep')
# Local contextuality
eval_f.add_pipe("local_contextuality")
# Grammaticality
eval_f.add_pipe('grammaticality')
# Diversity (lexical) self-bleu
eval_f.add_pipe('self-bleu')
# Diversity (lexical) moving mtld
eval_f.add_pipe('lexical_diversity')
# Complexity (lexical) unique words
eval_f.add_pipe('unique-words')
# Complexity (lexical) average word length
eval_f.add_pipe('avg-word-length')
# Complexity (syntactic)  average components
eval_f.add_pipe('average_components')
# Complexity (syntactic) dependency distance
eval_f.add_pipe('dependency_distance')
# Complexity (syntactic) syntactic tree depth
eval_f.add_pipe('syntactic_depth')
# Words before root
eval_f.add_pipe('wbr_average')

In [ ]:
def load_or_run_eval(eval_f, dataset, column, path_name, *, run=False):
	if Path(path_name).exists() and not run:
		df = pd.read_csv(path_name)
		if 'lexical_diversity' in df.columns:
			df['lexical_diversity'] = df['lexical_diversity'].apply(ast.literal_eval)
			
		return df

	dataset = eval_f.run_pipeline_on_df(dataset, column)
	dataset.to_csv(path_name, index=False)
	return dataset

In [ ]:
temp = all_datasets
for name, data in temp.items():
	print(f"Current method: {name}")
	generated_eval_results = load_or_run_eval(eval_f, data, 'story', f'results/metric_results/eval_results_{name}.csv')
	all_datasets[name] = generated_eval_results

### Plotting story quality

In [ ]:
nice_name = {'creative_perplexity_dep': 'Creative perplexity',
			 'local_contextuality': 'Local contextuality',
			 'grammaticality': 'Grammaticality',
			 'self-bleu': 'Self-bleu',
			 'lexical_diversity': 'Moving MTLD',
			 'unique-words': 'Size of vocabulary',
			 'avg-word-length': 'Average word length',
			 'average_components': 'Average components',
			 'dependency_distance': 'Dependency distance',
			 'syntactic_depth': 'Syntactic depth'
			 }

In [ ]:
def get_value_from_dictionary(x, key):
	new = []

	for item in x:
		new.append(item[key])
	return new

In [ ]:
metrics = ['creative_perplexity_dep', 'local_contextuality', 'grammaticality', 'self-bleu', 'lexical_diversity', 'unique-words', 'avg-word-length', 'average_components', 'dependency_distance', 'syntactic_depth']
story_metrics_means = {}

for name in all_datasets.keys():
	story_metrics_means[name] = defaultdict(float)

for name, data in all_datasets.items():
	for column in data.columns:
		if column in metrics:
			try:
				story_metrics_means[name][column] = data[column].mean()
			except:
				x = get_value_from_dictionary(data[column], 'moving_mtld')
				story_metrics_means[name][column] = np.mean(np.array(x))

In [ ]:
metrics = ['creative_perplexity_dep', 'local_contextuality', 'grammaticality', 'self-bleu', 'lexical_diversity', 'unique-words', 'avg-word-length', 'average_components', 'dependency_distance', 'syntactic_depth']
story_metrics_std = {}

for name in all_datasets.keys():
	story_metrics_std[name] = defaultdict(float)

for name, data in all_datasets.items():
	for column in data.columns:
		if column in metrics:
			try:
				story_metrics_std[name][column] = data[column].std()
			except:
				x = get_value_from_dictionary(data[column], 'moving_mtld')
				story_metrics_std[name][column] = np.std(np.array(x))

In [ ]:
df_story_means = pd.DataFrame.from_dict(story_metrics_means, orient="index")
df_story_means